# L5–L6 核方法（Kernel Methods）

## 严格保留第一版并加入参考书补充

本 Notebook 以最初生成的 Section 1–29 全文为底稿。每个 Section 中：

1. “第一版原文”完整保留原有推导、例子、解释和总结；
2. “补充与深化”加入 ltfp_book 与 MLbookSol 的对应内容；
3. 公式统一使用标准 Markdown：行内使用 `$...$`，独立公式使用 `$$...$$`；
4. 中文专业术语在 Section 开头给出对应英文。

<!-- FIRST_VERSION_PREAMBLE_START -->
下面以课件实际可见页面为准，把两讲连成一份可以独立学习的中文讲义。两份 slides 不是两个孤立主题，而是一条连续主线：

$$
\text{普通最小二乘的不稳定与过拟合}
\rightarrow
\text{正则化与偏差—方差}
\rightarrow
\text{非线性特征}
\rightarrow
\text{核技巧}
\rightarrow
\text{合法核的数学条件}
\rightarrow
\text{核 SVM、核岭回归、RKHS}
\rightarrow
\text{核回归与 Attention}.
$$

## 一、整体分块

| 讲次与页码 | 内容块 | 核心问题 |
|---|---|---|
| L5 p1–12 | 岭回归与正则化 | 为什么故意引入偏差，反而可能降低测试误差？ |
| L5 p13–20 | 非线性分类与特征映射 | 怎样把原空间中的非线性问题变成特征空间中的线性问题？ |
| L5 p21–25 | 特征从哪里来 | 手工设计、隐式使用，还是从数据学习？ |
| L5 p26–30 | 非线性 SVM 与核技巧 | 为什么 SVM 不必显式计算高维特征？ |
| L6 p2–13 | 核函数理论 | 什么函数才是真正合法的核？怎样构造和验证？ |
| L6 p14–20 | 核岭回归、RKHS、表示定理 | 为什么核模型最终只需要有限个训练样本系数？ |
| L6 p22–24 | KDE、核回归与 Attention | Attention 为什么可以看成一种学习到相似度的核回归？ |

下面详细展开。
<!-- FIRST_VERSION_PREAMBLE_END -->

## Section 1 — 普通最小二乘的问题（Problems with Ordinary Least Squares）

> **术语（Terminology）**：普通最小二乘（Ordinary Least Squares, OLS）；设计矩阵（Design Matrix）；正规方程（Normal Equations）；固定设计（Fixed Design）；随机设计（Random Design）

<!-- FIRST_VERSION_SECTION_1_START -->
---

# L5：正则化收尾与 Kernels I

## 1. 普通最小二乘的问题（p3）

训练数据为

$$
S=\{(x_1,y_1),\ldots,(x_N,y_N)\},
\qquad x_i\in\mathbb R^d,\ y_i\in\mathbb R.
$$

把样本按行放进设计矩阵：

$$
X=
\begin{bmatrix}
x_1^\top\\
\vdots\\
x_N^\top
\end{bmatrix}
\in\mathbb R^{N\times d},
\qquad
y\in\mathbb R^N,\quad w\in\mathbb R^d.
$$

普通最小二乘为

$$
\min_w L(w)
=
\sum_{i=1}^N(y_i-w^\top x_i)^2
=
\|Xw-y\|^2.
$$

求导：

$$
\nabla L(w)=2X^\top(Xw-y).
$$

令梯度为零，得到正规方程

$$
X^\top Xw=X^\top y.
$$

若 $X^\top X$ 可逆，则

$$
\boxed{
\hat w_{\mathrm{LS}}
=(X^\top X)^{-1}X^\top y.
}
$$

### $d<N$ 与 $d>N$ 分别意味着什么？

- 当 $d<N$ 时，样本数多于特征数，但这并不自动保证 $X^\top X$ 可逆。还必须有

  $$
  \operatorname{rank}(X)=d,
  $$

  即 $X$ 满列秩。

- 当 $d>N$ 时，

  $$
  \operatorname{rank}(X)\le N<d,
  $$

  所以 $X^\top X$ 必然奇异。最小二乘解通常不唯一。

- 即使 $d<N$，若特征高度共线，$X^\top X$ 也可能接近奇异。此时虽然理论上可以求逆，数值结果却会对很小的数据扰动极其敏感。

这引出本讲第一个问题：怎样既保证可解，又让模型更稳定？

---
<!-- FIRST_VERSION_SECTION_1_END -->

### 补充与深化（Supplement and Deepening）

> 来源：ltfp §3.2–3.5，PDF p.62–70。

#### 1. “线性模型”指对参数线性

一般形式是

$$
f_\theta(x)=\phi(x)^\top\theta.
$$

这里模型对参数 $\theta$ 是线性的，但对原始输入 $x$ 可以高度非线性。例如：

$$
\phi(x)=(1,x,x^2,\sin x,\ldots).
$$

所以“线性回归”不等于“只能拟合直线”；真正决定表达能力的是特征映射 $\phi$。

#### 2. OLS 是正交投影

令设计矩阵为 $\Phi\in\mathbb R^{n\times d}$。满列秩时：

$$
\hat\theta_{\rm OLS}
=(\Phi^\top\Phi)^{-1}\Phi^\top y.
$$

训练预测为

$$
\hat y=\Phi\hat\theta_{\rm OLS}
=P_\Phi y,
$$

其中

$$
P_\Phi=\Phi(\Phi^\top\Phi)^{-1}\Phi^\top
$$

是到 $\operatorname{im}(\Phi)$ 的正交投影。因此残差满足

$$
\Phi^\top(y-\hat y)=0.
$$

#### 3. 实际计算不要显式求逆

闭式公式主要用于理论分析。数值计算中直接形成

$$
(\Phi^\top\Phi)^{-1}
$$

会放大条件数并损失精度。实践中优先使用：

- QR 分解；
- Cholesky 分解；
- 共轭梯度；
- 梯度下降或随机梯度下降。

#### 4. 固定设计和随机设计必须区分

固定设计把 $x_1,\ldots,x_n$ 当作确定量，只重复生成噪声和标签。它衡量的是在这些已观察输入上的去噪能力。

随机设计则假设

$$
X\sim p_X,
$$

并衡量模型在新输入上的泛化。固定设计下的偏差—方差公式不能自动解释成随机设计泛化结论。

在线性真模型、满秩固定设计下：

$$
\mathbb E[R(\hat\theta_{\rm OLS})]-R^*
=\frac{\sigma^2d}{n}.
$$

若测试标签是在相同输入位置重新生成，则

$$
\mathbb E[\operatorname{MSE}_{\rm train}]
=\sigma^2\left(1-\frac dn\right),
$$

而

$$
\mathbb E[\operatorname{MSE}_{\rm test}]
=\sigma^2\left(1+\frac dn\right).
$$

所以训练误差平均低估测试误差

$$
\frac{2\sigma^2d}{n}.
$$

这定量解释了为什么参数增加会使训练误差下降，却可能导致泛化恶化。

---

## Section 2 — 从“加一个 $\lambda I$”到岭回归（Ridge Regression）

> **术语（Terminology）**：岭回归（Ridge Regression）；正则化（Regularization）；原始形式（Primal Form）；对偶形式（Dual Form）；条件数（Condition Number）

<!-- FIRST_VERSION_SECTION_2_START -->
## 2. 从“加一个 $\lambda I$”到岭回归（p4–5）

课件先提出一个看似纯代数的修补：

$$
X^\top X
\quad\longmapsto\quad
X^\top X+\lambda I,
\qquad \lambda>0.
$$

于是解变成

$$
\boxed{
\hat w_\lambda
=
(X^\top X+\lambda I)^{-1}X^\top y.
}
$$

### 为什么一定可逆？

因为

$$
v^\top X^\top Xv=\|Xv\|^2\ge0,
$$

所以 $X^\top X$ 是半正定矩阵。对任意非零 $v$，

$$
v^\top(X^\top X+\lambda I)v
=
\|Xv\|^2+\lambda\|v\|^2>0.
$$

因此 $X^\top X+\lambda I$ 是正定矩阵，一定可逆。

从特征值看，如果 $X^\top X$ 的特征值是

$$
\mu_1,\ldots,\mu_d,\qquad \mu_j\ge0,
$$

加入正则后变成

$$
\mu_j+\lambda>0.
$$

它既消除了零特征值，也抬高了非常小的特征值，从而改善条件数。

课件指出：使 $X^\top X$ 非奇异，正是 Hoerl 和 Kennard 在 1970 年提出岭回归时的原始动机。

### 这个代数修改对应什么优化问题？

岭回归写成

$$
\min_w
\frac12\|Xw-y\|^2+\lambda\|w\|^2.
$$

它同时要求：

1. 训练误差小；
2. 参数范数不要太大。

所以它不是单纯的“矩阵求逆技巧”，而是在估计中主动加入一种偏好：

> 在多个拟合程度相近的模型中，优先选择较小、较稳定的参数。

较大的参数常意味着：

- 对输入或训练样本的小扰动十分敏感；
- 在近共线方向上使用巨大且相互抵消的系数；
- 为了拟合噪声而让曲线剧烈波动。

### 关于课件中的常数 2

课件写的是

$$
\frac12\|Xw-y\|^2+\lambda\|w\|^2
$$

但给出 $X^\top X+\lambda I$ 的解。严格使用同一个 $\lambda$ 求导会得到 $X^\top X+2\lambda I$。

这不影响结论：通常把常数 2 吸收到超参数 $\lambda$ 的定义里。不同教材可能出现

$$
X^\top X+\lambda I,\qquad
X^\top X+2\lambda I,\qquad
X^\top X+N\lambda I,
$$

区别主要来自损失是否除以 $N$、正则项前是否有 $1/2$。

---
<!-- FIRST_VERSION_SECTION_2_END -->

### 补充与深化（Supplement and Deepening）

> 来源：ltfp §3.6–3.9，PDF p.72–84；ltfp §4.5.4–4.5.6，PDF p.112–121；MLbookSol p.34–38、p.65。

#### 1. Ridge 的 primal 与 dual 形式

对目标

$$
\min_\theta
\frac1n\|y-\Phi\theta\|_2^2+\lambda\|\theta\|_2^2,
$$

解为

$$
\boxed{
\hat\theta_\lambda
=(\Phi^\top\Phi+n\lambda I_d)^{-1}\Phi^\top y
}
$$

也等价于

$$
\boxed{
\hat\theta_\lambda
=\Phi^\top(\Phi\Phi^\top+n\lambda I_n)^{-1}y.
}
$$

因此：

- 当 $d\ll n$ 时，求解 $d\times d$ 的 primal 系统；
- 当 $d\gg n$ 时，求解 $n\times n$ 的 dual 系统；
- 当特征空间无限维时，只要能计算 $\Phi\Phi^\top$，dual 仍然可用。

这正是从岭回归过渡到核岭回归的桥梁。

## Section 3 — 正则化、测试误差与偏差—方差分解（Bias–Variance Decomposition）

> **术语（Terminology）**：偏差—方差分解（Bias–Variance Decomposition）；不可约噪声（Irreducible Noise）；近似误差（Approximation Error）；估计误差（Estimation Error）

<!-- FIRST_VERSION_SECTION_3_START -->
## 3. 正则化为什么能改善测试误差：偏差—方差分解（p6–8）

设真实观测模型为

$$
y=f(x)+\eta,
$$

其中 $f(x)$ 是真实的无噪声回归函数，$\eta$ 是零均值噪声。

训练过程得到随机预测器 $\hat f$。它是随机的，因为训练集、观测噪声甚至优化过程都可能变化。

关心的不是训练误差，而是

$$
\mathbb E\|y-\hat f\|^2.
$$

先加减 $f$：

$$
y-\hat f=(y-f)+(f-\hat f).
$$

展开平方：

$$
\|y-\hat f\|^2
=
\|y-f\|^2+\|f-\hat f\|^2
+2\langle y-f,f-\hat f\rangle.
$$

由于 $y-f=\eta$，在零均值和适当独立或不相关假设下，交叉项的期望为零：

$$
\mathbb E\|y-\hat f\|^2
=
\mathbb E\|\eta\|^2
+
\mathbb E\|f-\hat f\|^2.
$$

然后在第二项中加减平均预测器 $\mathbb E\hat f$：

$$
f-\hat f
=
f-\mathbb E\hat f+\mathbb E\hat f-\hat f.
$$

再次展开并取期望：

$$
\mathbb E\|f-\hat f\|^2
=
\|f-\mathbb E\hat f\|^2
+
\mathbb E\|\hat f-\mathbb E\hat f\|^2.
$$

最终得到

$$
\boxed{
\mathbb E\|y-\hat f\|^2
=
\underbrace{\mathbb E\|\eta\|^2}_{\text{不可约噪声}}
+
\underbrace{\|f-\mathbb E\hat f\|^2}_{\text{偏差平方}}
+
\underbrace{\mathbb E\|\hat f-\mathbb E\hat f\|^2}_{\text{方差}}.
}
$$

三项分别表示：

- 不可约噪声：数据生成机制本身的随机性；
- 偏差平方：重复训练后，平均模型离真实函数多远；
- 方差：更换训练集时，模型变化有多大。

### 为什么有偏模型可能更好？

普通最小二乘在经典条件下是 BLUE：

> Best Linear Unbiased Estimator，即最佳线性无偏估计量。

这就是 Gauss–Markov 定理。但“最佳”只是在“线性且无偏”的估计量中方差最低，不代表：

- 它在所有有偏估计量中最好；
- 它的测试均方误差一定最低；
- 它在所有非线性估计量中最好。

岭回归主动引入偏差，但可以显著降低方差。只要方差下降超过偏差增加，测试误差就会降低。

---
<!-- FIRST_VERSION_SECTION_3_END -->

### 补充与深化（Supplement and Deepening）

> 来源：ltfp §3.6–3.9，PDF p.72–84；ltfp §4.5.4–4.5.6，PDF p.112–121；MLbookSol p.34–38、p.65。

#### 8. 两种不同的误差分解

不要把下列两个分解当成同一个公式：

平方损失的 bias–variance：

$$
\text{总误差}
=
\text{不可约噪声}
+\text{偏差平方}
+\text{方差}.
$$

学习理论中的 approximation–estimation：

$$
L_D(\hat h)-L_D^*
=
\underbrace{L_D(h_{\mathcal H}^*)-L_D^*}_{\text{近似误差}}
+
\underbrace{L_D(\hat h)-L_D(h_{\mathcal H}^*)}_{\text{估计误差}}.
$$

扩大模型类通常降低近似误差，却提高估计误差。

---

## Section 4 — 岭回归的偏差（Bias of Ridge Regression）

> **术语（Terminology）**：估计偏差（Estimator Bias）；参数协方差（Parameter Covariance）；谱收缩（Spectral Shrinkage）

<!-- FIRST_VERSION_SECTION_4_START -->
## 4. 岭回归确实是有偏的（p9）

采用平均损失的尺度：

$$
\min_w
\frac1N\|Xw-y\|^2+\lambda\|w\|^2.
$$

解为

$$
\hat w_{\mathrm{RR}}
=
(X^\top X+N\lambda I)^{-1}X^\top y.
$$

令

$$
M=X^\top X.
$$

普通最小二乘为

$$
\hat w_{\mathrm{LS}}=M^{-1}X^\top y.
$$

于是

$$
\begin{aligned}
\hat w_{\mathrm{RR}}
&=(M+N\lambda I)^{-1}X^\top y\\
&=(M+N\lambda I)^{-1}M\hat w_{\mathrm{LS}}\\
&=(I+N\lambda M^{-1})^{-1}\hat w_{\mathrm{LS}}.
\end{aligned}
$$

定义

$$
A=(I+N\lambda M^{-1})^{-1},
$$

则

$$
\hat w_{\mathrm{RR}}=A\hat w_{\mathrm{LS}}.
$$

假设

$$
y=Xw_{\mathrm{true}}+\eta,\qquad
\mathbb E\eta=0.
$$

由于 OLS 无偏，

$$
\mathbb E\hat w_{\mathrm{LS}}=w_{\mathrm{true}}.
$$

所以

$$
\boxed{
\mathbb E\hat w_{\mathrm{RR}}
=
Aw_{\mathrm{true}}
\ne w_{\mathrm{true}}
}
$$

通常在 $\lambda>0$ 时成立。

偏差向量为

$$
\operatorname{Bias}(\hat w_{\mathrm{RR}})
=
(A-I)w_{\mathrm{true}}
=
-N\lambda(M+N\lambda I)^{-1}w_{\mathrm{true}}.
$$

### 课件留下的方差练习

若

$$
\operatorname{Var}(\eta)=\sigma^2I,
$$

则

$$
\operatorname{Var}(\hat w_{\mathrm{LS}})
=
\sigma^2M^{-1}.
$$

因此

$$
\boxed{
\operatorname{Var}(\hat w_{\mathrm{RR}})
=
A\operatorname{Var}(\hat w_{\mathrm{LS}})A^\top
=
\sigma^2(M+N\lambda I)^{-1}
M
(M+N\lambda I)^{-1}.
}
$$

若 $M$ 的某个特征方向对应特征值 $\mu_j$，那么：

- OLS 方差：

  $$
  \frac{\sigma^2}{\mu_j};
  $$

- 岭回归方差：

  $$
  \frac{\sigma^2\mu_j}{(\mu_j+N\lambda)^2};
  $$

- 均值收缩系数：

  $$
  \frac{\mu_j}{\mu_j+N\lambda}.
  $$

当 $\mu_j$ 很小时，OLS 方差可能巨大；岭回归恰好会最强烈地压缩这些弱信息、病态方向。

---
<!-- FIRST_VERSION_SECTION_4_END -->

### 补充与深化（Supplement and Deepening）

> 来源：ltfp §3.6–3.9，PDF p.72–84；ltfp §4.5.4–4.5.6，PDF p.112–121；MLbookSol p.34–38、p.65。

#### 2. Ridge 的精确谱偏差—方差

令

$$
\widehat\Sigma=\frac1n\Phi^\top\Phi
=U\operatorname{diag}(\mu_1,\ldots,\mu_d)U^\top,
\qquad
a_j=u_j^\top\theta_*.
$$

固定设计下：

$$
\begin{aligned}
\mathbb E[R(\hat\theta_\lambda)]-R^*
={}&
\lambda^2\theta_*^\top
(\widehat\Sigma+\lambda I)^{-2}
\widehat\Sigma\theta_*\\
&+\frac{\sigma^2}{n}
\operatorname{tr}\!\left[
\widehat\Sigma^2(\widehat\Sigma+\lambda I)^{-2}
\right].
\end{aligned}
$$

在各特征方向上：

$$
\operatorname{Bias}^2
=
\sum_j
\frac{\lambda^2\mu_j}{(\mu_j+\lambda)^2}a_j^2,
$$

$$
\operatorname{Variance}
=
\frac{\sigma^2}{n}
\sum_j
\left(\frac{\mu_j}{\mu_j+\lambda}\right)^2.
$$

因此 Ridge 不是简单地“把所有系数缩小相同比例”，而是：

- $\mu_j\gg\lambda$：数据充分支持该方向，保留较多；
- $\mu_j\ll\lambda$：数据几乎不支持该方向，被强烈抑制。

## Section 5 — $\lambda$ 与模型复杂度（Model Complexity）

> **术语（Terminology）**：模型复杂度（Model Complexity）；有效自由度（Effective Degrees of Freedom）；模型选择（Model Selection）

<!-- FIRST_VERSION_SECTION_5_START -->
## 5. $\lambda$ 控制模型复杂度（p10）

一般趋势是：

- 模型越复杂：偏差低、方差高；
- 模型越简单：偏差高、方差低。

在岭回归中，$\lambda$ 是控制旋钮：

- $\lambda=0$：OLS，自由度最高；
- $\lambda$ 较小：弱收缩，偏差较低、方差较高；
- $\lambda$ 较大：参数被压向零，偏差较高、方差较低。

课件中的图展示：

- 偏差平方随 $\lambda$ 增大而上升；
- 方差随 $\lambda$ 增大而下降；
- 二者之和通常存在最低点；
- 测试误差的最低点与这个最低点接近。

所以 $\lambda$ 通常不能只看训练误差选择，而需要验证集、交叉验证等方法。

---
<!-- FIRST_VERSION_SECTION_5_END -->

### 补充与深化（Supplement and Deepening）

> 来源：ltfp §3.6–3.9，PDF p.72–84；ltfp §4.5.4–4.5.6，PDF p.112–121；MLbookSol p.34–38、p.65。

#### 3. 两种“有效自由度”

书中控制固定设计方差的量是

$$
df_2(\lambda)
=
\sum_j
\left(\frac{\mu_j}{\mu_j+\lambda}\right)^2.
$$

统计文献还常把平滑矩阵的迹

$$
df_1(\lambda)
=
\sum_j
\frac{\mu_j}{\mu_j+\lambda}
$$

称为有效自由度。

二者不要混淆：

- $df_1=\operatorname{tr}(H_\lambda)$：模型的软参数数量；
- $df_2=\operatorname{tr}(H_\lambda^2)$：直接出现在预测方差中。

#### 4. 一个不显含维数的风险界

令

$$
\operatorname{tr}(\widehat\Sigma)
=\frac1n\sum_i\|\phi(x_i)\|^2.
$$

平衡偏差和方差上界可得

$$
\lambda_*
=
\frac{\sigma\sqrt{\operatorname{tr}(\widehat\Sigma)}}
{\|\theta_*\|\sqrt n},
$$

以及

$$
\mathbb E[R(\hat\theta_{\lambda_*})]-R^*
\le
\frac{
\sigma\|\theta_*\|
\sqrt{\operatorname{tr}(\widehat\Sigma)}
}{\sqrt n}.
$$

该界不显式依赖 $d$，说明维数本身并不总是复杂度的最佳度量；参数范数与特征范数往往更重要。

但 $\lambda_*$ 依赖未知的 $\sigma,\theta_*$，实践中仍需交叉验证。

## Section 6 — 正则化的完整动机（Motivations for Regularization）

> **术语（Terminology）**：稳定性（Stability）；结构风险最小化（Structural Risk Minimization, SRM）；最大后验估计（Maximum A Posteriori, MAP）；软阈值（Soft Thresholding）

<!-- FIRST_VERSION_SECTION_6_START -->
## 6. 正则化的完整动机（p11–12）

正则化不只是“防止矩阵不可逆”，课件给出四个更一般的理由。

### 6.1 欠定问题

若 $d>N$，$X^\top X$ 必然奇异，而岭回归仍给出唯一解。

### 6.2 病态与数值稳定性

即便 $N\ge d$，矩阵也可能接近奇异，导致：

- 难以准确求解；
- 浮点误差被放大；
- 数据微小变化引起参数巨大变化；
- 模型缺乏稳定性与鲁棒性。

### 6.3 防止拟合噪声

高阶特征或复杂模型可以让训练误差很低，却可能追随每个噪声点，导致未见数据上的误差变大。

### 6.4 注入先验结构

如果知道“合理模型应该是什么样”，可以通过：

- 贝叶斯先验；
- 参数约束；
- 惩罚项；

把这种结构加入学习问题。

一般正则化形式为

$$
\boxed{
\min_w
\frac1N\sum_i(y_i-w^\top x_i)^2
+
\lambda\Omega(w).
}
$$

其中：

- 经验风险负责拟合数据；
- $\Omega(w)$ 决定偏好什么样的模型；
- $\lambda$ 决定数据拟合与结构偏好的相对强度。

例如：

- $\Omega(w)=\|w\|_2^2$：偏好小而平滑的参数；
- $\Omega(w)=\|w\|_1$：通常鼓励稀疏；
- 其他正则器可表达低秩、分组、平滑等结构。

课件还强调：正则化不一定显式写成 $\lambda\Omega(w)$。模型架构、优化算法、计算限制和数据增强等，都可能形成隐式正则化。
<!-- FIRST_VERSION_SECTION_6_END -->

### 补充与深化（Supplement and Deepening）

> 来源：ltfp §3.6–3.9，PDF p.72–84；ltfp §4.5.4–4.5.6，PDF p.112–121；MLbookSol p.34–38、p.65。

#### 5. Ridge 的贝叶斯解释

若

$$
\theta_*\sim
\mathcal N\!\left(
0,\frac{\sigma^2}{n\lambda}I
\right),
\qquad
y\mid\theta_*
\sim\mathcal N(\Phi\theta_*,\sigma^2I),
$$

则 Ridge 同时是：

- 后验均值；
- 后验众数 MAP。

因此 L2 正则化可以解释为“参数在零附近、没有特别大的方向”的高斯先验。

#### 6. 正则化与稳定性

MLbookSol 给出一个一般结论：若算法具有 $\epsilon_{\rm stab}$ 的平均稳定性，同时又是 $\epsilon_{\rm ERM}$-近似 ERM，则

$$
\boxed{
\mathbb E[
L_D(A(S))-L_D(h^*)
]
\le
\epsilon_{\rm stab}+\epsilon_{\rm ERM}.
}
$$

对 L2 正则化、Lipschitz 损失，稳定性项典型地满足

$$
\epsilon_{\rm stab}
=
O\!\left(
\frac{G^2R^2}{\lambda n}
\right).
$$

所以：

- 增大 $\lambda$：算法更稳定，估计误差下降；
- 但正则化偏差增大。

这是正则化为何改善泛化的另一种解释，不仅仅是“降低参数方差”。

#### 7. L1 与 L2 的根本区别

一维问题

$$
\min_w
\frac12w^2-xw+\lambda|w|
$$

的解是软阈值：

$$
\boxed{
w=\operatorname{sign}(x)(|x|-\lambda)_+.
}
$$

因此 L1 会把较小的系数精确压到零；L2 一般只会连续缩小而不会产生精确零。这解释了 Lasso 的特征选择能力。

## Section 7 — 非线性分类的两条路线（Two Routes to Nonlinear Classification）

> **术语（Terminology）**：非线性分类（Nonlinear Classification）；特征映射（Feature Map）；有限假设类（Finite Hypothesis Class）

<!-- FIRST_VERSION_SECTION_7_START -->
---

# L5 第二部分：从非线性分类到特征映射

## 7. 处理非线性分类的两条路线（p13–17）

两种思路是：

1. 在原始输入上直接使用非线性分类器；
2. 先做非线性特征映射

   $$
   x\mapsto\phi(x),
   $$

   再使用线性分类器：

   $$
   h(x)=\operatorname{sgn}
   \bigl(w^\top\phi(x)+w_0\bigr).
   $$

第二种方法在特征空间里是线性的，但在原空间里会形成非线性边界。

### p14：椭圆分类器

分类规则为

$$
(x_1-c_1)^2+b(x_2-c_2)^2\le1
$$

时预测 $+1$，否则预测 $-1$。

这是原空间中的椭圆边界。展开：

$$
x_1^2+b x_2^2
-2c_1x_1-2bc_2x_2
+c_1^2+bc_2^2-1\le0.
$$

它其实是在特征

$$
\phi(x)=(1,x_1,x_2,x_1^2,x_2^2)
$$

上做线性判别。

### p15：多个超平面

一条直线不能圈出中间区域时，可以组合多个半空间，形成多边形或分段线性决策区域。

每个边界本身是线性的，但“同时满足多个线性条件”的逻辑组合已经是非线性分类器。

### p16：kNN

课件展示 15-nearest-neighbor 的不规则决策边界。

kNN 不显式建立解析边界，而是：

1. 找到新点附近的 $k$ 个训练样本；
2. 用邻居多数类别预测。

边界由局部样本分布决定，因此高度非线性。

通常：

- 小 $k$：边界复杂，低偏差、高方差；
- 大 $k$：边界平滑，高偏差、低方差。

---
<!-- FIRST_VERSION_SECTION_7_END -->

### 补充与深化（Supplement and Deepening）

> 来源：ltfp §7.3.1，PDF p.202–203；MLbookSol p.41。

#### 2. 任意有限假设类都可以线性化

设

$$
\mathcal H=\{h_1,\ldots,h_M\}.
$$

可以定义

$$
\phi(x)=
(\mathbf1\{h_1(x)=1\},\ldots,
\mathbf1\{h_M(x)=1\},1).
$$

对第 $j$ 个假设取

$$
w_j=(2e_j,-1),
$$

则

$$
\langle w_j,\phi(x)\rangle
=
2\mathbf1\{h_j(x)=1\}-1
=h_j(x).
$$

所以问题不在于“能否把非线性规则线性化”，而在于显式特征维数可能极大，进而需要核技巧或近似特征。

---

## Section 8 — 手工构造非线性特征（Handcrafted Nonlinear Features）

> **术语（Terminology）**：手工特征（Handcrafted Features）；多项式特征（Polynomial Features）；交互项（Interaction Term）

<!-- FIRST_VERSION_SECTION_8_START -->
## 8. 手工构造非线性特征（p18–20）

### 8.1 一维的 V 形映射

原数据中，中间点是一类，两端点是另一类。一个一维阈值无法同时把左右两端归为同一类。

使用

$$
\boxed{
\phi(x)=(x,|x|).
}
$$

映射后：

- 中间点的 $|x|$ 小，位于 V 形底部；
- 两端点的 $|x|$ 大，位于 V 形上部。

于是二维空间中的一条水平直线就能分隔两类。

### 8.2 XOR 式数据与交互项

原二维空间中，对角象限属于同一类别，无法用一条直线分开。

映射为

$$
\boxed{
\phi(x_1,x_2)
=
(x_1,x_1x_2,x_2).
}
$$

关键是 $x_1x_2$：

- $x_1,x_2$ 同号时，$x_1x_2>0$；
- 异号时，$x_1x_2<0$。

因此新坐标的正负直接区分两种对角结构，在三维空间中可以线性分离。

### 8.3 特征维数为什么会爆炸？

对于 $p$ 个原始变量：

- 线性特征：

  $$
  (x_1,\ldots,x_p),
  $$

  维数为 $p$。

- 所有不重复的二次单项式 $x_ix_j,\ i\le j$，维数为

  $$
  \boxed{\frac{p(p+1)}2}.
  $$

- 所有总次数不超过 $d$ 的多项式单项式：

  - 包含常数项：

    $$
    \binom{p+d}{d};
    $$

  - 不包含常数项：

    $$
    \boxed{\binom{p+d}{d}-1}.
    $$

特征数量会随维度和次数组合爆炸；有些有用的特征空间甚至是无限维。这就是后面不显式构造 $\phi(x)$ 的原因。

---
<!-- FIRST_VERSION_SECTION_8_END -->

### 补充与深化（Supplement and Deepening）

> 来源：ltfp §7.3.1，PDF p.202–203；MLbookSol p.41。

#### 1. 多项式核的精确特征维数

齐次多项式核

$$
k_s(x,z)=(x^\top z)^s
$$

对应所有总次数恰好为 $s$ 的单项式，最小显式特征维数为

$$
\boxed{
\binom{d+s-1}{s}.
}
$$

非齐次核

$$
k_s(x,z)=(1+x^\top z)^s
$$

包含所有次数不超过 $s$ 的单项式，维数为

$$
\boxed{
\binom{d+s}{s}.
}
$$

原笔记中的 $d^s$ 可以理解为尚未合并重复单项式的有序张量维数；合并后才得到上述组合数。

## Section 9 — 特征的来源（Sources of Features）

> **术语（Terminology）**：显式特征（Explicit Features）；隐式特征（Implicit Features）；学习特征（Learned Features）；归纳偏置（Inductive Bias）

<!-- FIRST_VERSION_SECTION_9_START -->
## 9. 特征可以怎样获得（p21–25）

课件区分三种范式。

### 显式特征提取

$$
\text{原始数据}
\rightarrow
\text{特征向量}
\rightarrow
\text{分类器}.
$$

例如把邮件转换成词袋、词频或 TF-IDF 向量。

### 隐式使用非线性特征

使用一个非线性分类器，但把它理解为某个未知或未显式构造的特征空间中的线性方法。

核方法属于这一类。

### 从数据学习特征

不手工固定 $\phi$，而让训练过程本身学习表示。深度神经网络是典型例子。

本讲接下来选择的是：

> 隐式指定特征映射，只通过内积访问它。
<!-- FIRST_VERSION_SECTION_9_END -->

## Section 10 — 硬间隔与软间隔支持向量机（Hard- and Soft-Margin SVM）

> **术语（Terminology）**：支持向量机（Support Vector Machine, SVM）；硬间隔（Hard Margin）；软间隔（Soft Margin）；铰链损失（Hinge Loss）

<!-- FIRST_VERSION_SECTION_10_START -->
---

# L5 第三部分：从 SVM 到核技巧

## 10. 硬间隔和软间隔 SVM（p26–27）

硬间隔 SVM：

$$
\min_{w,w_0}\frac12\|w\|^2
$$

满足

$$
y_i(w^\top x_i+w_0)\ge1.
$$

最小化 $\|w\|$ 等价于最大化几何间隔。硬间隔要求训练数据严格可分。

软间隔 SVM 引入 $\xi_i\ge0$：

$$
\boxed{
\min_{w,w_0,\xi}
\frac12\|w\|^2+C\sum_i\xi_i
}
$$

满足

$$
y_i(w^\top x_i+w_0)\ge1-\xi_i,
\qquad
\xi_i\ge0.
$$

解释：

- $\xi_i=0$：满足标准间隔；
- $0<\xi_i\le1$：分类正确，但位于间隔内部；
- $\xi_i>1$：被错误分类。

$C$ 越大，越不容忍训练违例；$C$ 越小，允许更多违例以换取更宽间隔和更强正则化。

---
<!-- FIRST_VERSION_SECTION_10_END -->

### 补充与深化（Supplement and Deepening）

> 来源：ltfp §4.1.2，PDF p.90–95；ltfp §12.1.2，PDF p.362–364；MLbookSol p.39–44。

#### 1. 软间隔 SVM 中 $C$ 与 $\lambda$ 的关系

标准形式

$$
\min_{w,b,\xi}
\frac12\|w\|^2+C\sum_i\xi_i
$$

与正则化 hinge-risk 形式

$$
\min_{w,b}
\frac1n\sum_i
(1-y_i(w^\top x_i+b))_+
+\frac\lambda2\|w\|^2
$$

满足

$$
\boxed{\lambda=\frac1{nC}}.
$$

所以：

- 大 $C$：更重视训练违例，正则化弱；
- 小 $C$：允许更多违例，正则化强。

#### 2. SVM 对偶和支持向量分类

对偶问题为

$$
\max_\alpha
\sum_i\alpha_i
-\frac12
\sum_{i,j}
\alpha_i\alpha_jy_iy_jK(x_i,x_j),
$$

满足

$$
0\le\alpha_i\le C,
\qquad
\sum_i\alpha_i y_i=0.
$$

KKT 条件给出：

- $y_if(x_i)>1\Rightarrow\alpha_i=0$：不影响最终预测；
- $0<\alpha_i<C\Rightarrow y_if(x_i)=1$：位于间隔边界；
- $\alpha_i=C$：点可能位于间隔内部或被误分类。

支持向量稀疏性有计算价值，但高维问题中支持向量数可能与 $n$ 同阶，不能仅靠“支持向量少”解释 SVM 的统计性能。

#### 3. 线性可分不代表软 SVM 等于硬 SVM

MLbookSol 给出一个反例：存在一个离决策边界极近的可分样本。硬间隔为了正确覆盖它，需要非常大的 $\|w\|$；有限 $C$ 的软 SVM 可能宁愿违反该点，以换取小得多的参数范数。

因此：

> 可分性只保证硬间隔问题可行；固定有限 $C$ 的软 SVM 不一定产生硬间隔解。

#### 4. Hinge loss 的 Lipschitz 性

若 $\|x\|\le R$，则

$$
\ell(w;(x,y))
=(1-yw^\top x)_+
$$

满足

$$
|\ell(w_1)-\ell(w_2)|
\le
R\|w_1-w_2\|.
$$

这使参数稳定性可以直接转化为损失稳定性。

此外，hinge loss 具有分类校准性质：

$$
R_{01}(f)-R_{01}^*
\le
R_{\rm hinge}(f)-R_{\rm hinge}^*.
$$

即把 hinge 风险优化到接近最优，会相应控制 0–1 分类风险。

#### 5. 间隔还控制感知机更新次数

若 $\|x_i\|\le\rho$，数据可由单位向量以间隔 $\gamma$ 分开，则感知机犯错次数满足

$$
\boxed{
T_{\rm mistake}\le
\left(\frac{\rho}{\gamma}\right)^2.
}
$$

因此大间隔不仅有泛化意义，也会加快感知机收敛。

#### 7. Logistic GD 的隐式最大间隔偏置

在线性可分数据上，无正则 logistic loss 的下确界为零，但不存在有限极小点。梯度下降会满足

$$
\|\theta_t\|\to\infty,
$$

然而方向收敛：

$$
\frac{\theta_t}{\|\theta_t\|}
\longrightarrow
\arg\max_{\|\eta\|\le1}
\min_i y_i x_i^\top\eta.
$$

右侧恰是 hard-margin SVM 的最大间隔方向。这说明优化算法本身也可以产生隐式正则化。

---

## Section 11 — 非线性特征空间中的 SVM（SVM in a Nonlinear Feature Space）

> **术语（Terminology）**：几何间隔（Geometric Margin）；松弛变量（Slack Variables）；KKT 条件（Karush–Kuhn–Tucker Conditions）；支持向量（Support Vector）

<!-- FIRST_VERSION_SECTION_11_START -->
## 11. 将 SVM 放入非线性特征空间（p28）

把 $x_i$ 替换为 $\phi(x_i)$：

$$
\min_{w,w_0,\xi}
\frac12\|w\|^2+C\sum_i\xi_i
$$

满足

$$
y_i\bigl(\langle w,\phi(x_i)\rangle+w_0\bigr)
\ge1-\xi_i.
$$

其对偶问题可以写为

$$
\max_\alpha
\sum_i\alpha_i
-
\frac12
\sum_{i,j}
\alpha_i\alpha_jy_iy_j
\langle\phi(x_i),\phi(x_j)\rangle
$$

满足

$$
\sum_i\alpha_i y_i=0,
\qquad
0\le\alpha_i\le C.
$$

由拉格朗日驻点条件：

$$
\boxed{
w=\sum_i\alpha_i y_i\phi(x_i).
}
$$

所以

$$
\begin{aligned}
\langle w,\phi(x)\rangle
&=
\sum_i\alpha_i y_i
\langle\phi(x_i),\phi(x)\rangle.
\end{aligned}
$$

关键发现：

> SVM 不需要知道 $\phi(x)$ 的具体坐标，只需要计算特征空间内积。

---
<!-- FIRST_VERSION_SECTION_11_END -->

## Section 12 — 核技巧（Kernel Trick）

> **术语（Terminology）**：核技巧（Kernel Trick）；核感知机（Kernel Perceptron）；特征空间内积（Feature-Space Inner Product）

<!-- FIRST_VERSION_SECTION_12_START -->
## 12. 核技巧（p29–30）

如果存在函数

$$
\boxed{
k(x,x')
=
\langle\phi(x),\phi(x')\rangle,
}
$$

那么所有特征空间内积都可以直接替换成核值：

$$
\langle w,\phi(x)\rangle
=
\sum_i\alpha_i y_i k(x_i,x).
$$

核 SVM 的最终分类器为

$$
\boxed{
h(x)
=
\operatorname{sgn}
\left(
\sum_i\alpha_i y_i k(x_i,x)+w_0
\right).
}
$$

只有 $\alpha_i>0$ 的点出现在最终决策函数中，它们就是支持向量。

实践中不需要：

- 显式构造高维 $\phi(x)$；
- 显式存储特征空间向量 $w$。

只需要：

- 训练样本；
- 系数 $\alpha_i$；
- 标签 $y_i$；
- 偏置 $w_0$；
- 核函数。

课件所说“能够计算 $w$”应理解为获得其隐式展开

$$
w=\sum_i\alpha_i y_i\phi(x_i),
$$

而不是一定形成 $w$ 的坐标。

这不只适用于 SVM。只要某方法的解具有

$$
w=\sum_i a_i\phi(x_i)
$$

的形式，就可能核化。感知机也是如此，因为每次更新都是把某个训练样本特征加到 $w$ 中。
<!-- FIRST_VERSION_SECTION_12_END -->

### 补充与深化（Supplement and Deepening）

> 来源：ltfp §4.1.2，PDF p.90–95；ltfp §12.1.2，PDF p.362–364；MLbookSol p.39–44。

#### 6. 核感知机

保存系数而非显式权重：

$$
w^{(t)}
=
\sum_i\alpha_i^{(t)}\psi(x_i).
$$

若样本 $i$ 被误分类，则

$$
y_i\sum_j\alpha_jK(x_j,x_i)\le0,
$$

并更新

$$
\alpha_i\leftarrow\alpha_i+y_i.
$$

预测为

$$
h(x)
=
\operatorname{sign}
\left(
\sum_i\alpha_iK(x_i,x)
\right).
$$

这里标签已吸收到有符号系数 $\alpha_i$ 中；不要再额外乘一次 $y_i$。

## Section 13 — 核技巧回顾与参考资料（Kernel-Trick Review and References）

> **术语（Terminology）**：核函数（Kernel Function）；核 SVM（Kernel SVM）；无限维特征（Infinite-Dimensional Features）

<!-- FIRST_VERSION_SECTION_13_START -->
---

# L6：Kernels II

L6 开始回答 L5 留下的核心问题：

> 不能随便拿一个“相似度”当作核。什么函数确实等于某个特征空间中的内积？

## 13. 核技巧回顾与参考资料（p2–3）

核函数的基本目标是

$$
k(x,z)=\langle\phi(x),\phi(z)\rangle.
$$

如果计算 $k(x,z)$ 比显式构造 $\phi(x)$ 便宜，就能够在高维或无限维空间中高效学习。

课件推荐：

- Schölkopf 与 Smola 的 *Learning with Kernels*；
- Berg、Christensen、Ressel 的 *Harmonic Analysis on Semigroups*。

第一本偏机器学习核方法，第二本深入正定函数与调和分析。

---
<!-- FIRST_VERSION_SECTION_13_END -->

## Section 14 — 多项式核的显式例子（Explicit Polynomial-Kernel Example）

> **术语（Terminology）**：多项式核（Polynomial Kernel）；张量特征（Tensor Features）；多重指标（Multi-Index）

<!-- FIRST_VERSION_SECTION_14_START -->
## 14. 多项式核的显式例子（p4）

定义

$$
\phi:\mathbb R^2\to\mathbb R^3,
\qquad
\phi(x)=
(x_1^2,\sqrt2x_1x_2,x_2^2).
$$

那么

$$
\begin{aligned}
\langle\phi(x),\phi(z)\rangle
&=
x_1^2z_1^2
+2x_1x_2z_1z_2
+x_2^2z_2^2\\
&=
(x_1z_1+x_2z_2)^2\\
&=
\langle x,z\rangle^2.
\end{aligned}
$$

所以

$$
k(x,z)=\langle x,z\rangle^2
$$

隐式对应一个二次特征空间。

一般地，在 $\mathbb R^d$ 上

$$
k_m(x,z)=\langle x,z\rangle^m.
$$

课件采用按有序指标排列的特征：

$$
\phi_m(x)
=
\bigl(
x_{i_1}\cdots x_{i_m}
\bigr)_{1\le i_1,\ldots,i_m\le d}.
$$

它有 $d^m$ 个坐标，并且

$$
\begin{aligned}
\langle\phi_m(x),\phi_m(z)\rangle
&=
\sum_{i_1,\ldots,i_m}
\prod_{r=1}^m x_{i_r}z_{i_r}\\
&=
\left(\sum_{j=1}^d x_jz_j\right)^m\\
&=
\langle x,z\rangle^m.
\end{aligned}
$$

显式特征需要处理 $d^m$ 个坐标，而核只需 $O(d)$ 计算原始内积，再取 $m$ 次幂。

### 为什么课件说这个 $\phi$ 与前面的 $\phi$ 不同？

当 $d=2,m=2$ 时，有序构造是

$$
(x_1^2,x_1x_2,x_2x_1,x_2^2),
$$

而压缩后的构造是

$$
(x_1^2,\sqrt2x_1x_2,x_2^2).
$$

两种特征映射不同，却生成同一个核。这说明：

$$
\boxed{\text{特征映射不唯一。}}
$$

若合并重复单项式，恰好 $m$ 次的标准多项式特征维数为

$$
\binom{d+m-1}{m}.
$$

---
<!-- FIRST_VERSION_SECTION_14_END -->

## Section 15 — 合法核的条件（Requirements for a Valid Kernel）

> **术语（Terminology）**：合法核（Valid Kernel）；对称性（Symmetry）；正半定（Positive Semidefinite, PSD）；Gram 矩阵（Gram Matrix）

<!-- FIRST_VERSION_SECTION_15_START -->
## 15. 一个好的核需要满足什么（p5）

### 可计算性

$k(x,z)$ 应能高效计算，通常要比显式高维特征内积便宜。

### 对任务有意义

数学上合法的核不一定适合具体任务。核定义了：

- 哪些输入被认为相似；
- 模型偏好什么样的函数；
- 哪种变化被认为重要或不重要。

### 对称性

因为内积对称：

$$
k(x,z)=k(z,x).
$$

### 正半定性

真正关键的条件是：对任意有限样本 $x_1,\ldots,x_m$ 和任意系数 $c_1,\ldots,c_m$，

$$
\boxed{
\sum_{i,j=1}^m
c_ic_jk(x_i,x_j)\ge0.
}
$$

若定义 Gram 矩阵

$$
K_{ij}=k(x_i,x_j),
$$

则条件等价于

$$
c^\top Kc\ge0
\quad\text{对所有 }c.
$$

课件常把这种核称为 positive definite 或 Mercer kernel，但矩阵实际上允许零特征值，因此通常更精确地称为 positive semidefinite kernel。

如果存在特征映射，那么

$$
c^\top Kc
=
\sum_{i,j}c_ic_j
\langle\phi(x_i),\phi(x_j)\rangle
=
\left\|
\sum_i c_i\phi(x_i)
\right\|^2
\ge0.
$$

反过来，如果所有有限 Gram 矩阵都正半定，就可以构造相应的 Hilbert 特征空间。

---
<!-- FIRST_VERSION_SECTION_15_END -->

### 补充与深化（Supplement and Deepening）

> 来源：ltfp §7.2–7.3，PDF p.197–212；MLbookSol p.42–44。

#### 1. 两个非常直观的合法核

在 $\{1,\ldots,N\}$ 上：

$$
K(i,j)=\min(i,j).
$$

取

$$
\psi(j)=
(\underbrace{1,\ldots,1}_{j},
0,\ldots,0),
$$

就有

$$
K(i,j)=\langle\psi(i),\psi(j)\rangle.
$$

对集合 $E,E'\subseteq[d]$：

$$
\boxed{
K(E,E')=|E\cap E'|.
}
$$

因为集合指示向量满足

$$
|E\cap E'|
=
\langle\mathbf1_E,\mathbf1_{E'}\rangle.
$$

这说明输入不需要是欧氏向量；字符串、集合、文本、树、图等都可以直接定义核。

## Section 16 — 核函数的代数（Kernel Algebra）

> **术语（Terminology）**：闭包性质（Closure Properties）；直和（Direct Sum）；张量积（Tensor Product）；Schur 乘积定理（Schur Product Theorem）

<!-- FIRST_VERSION_SECTION_16_START -->
## 16. 核函数的代数（p6–9）

如果 $k_1,k_2$ 是合法核，则以下仍是核。

### 非负缩放

$$
\tilde k(x,z)=c\,k_1(x,z),
\qquad c\ge0.
$$

对应特征映射

$$
\tilde\phi(x)=\sqrt c\,\phi(x).
$$

### 求和

$$
k(x,z)=k_1(x,z)+k_2(x,z).
$$

对应直和特征

$$
\phi(x)=\phi_1(x)\oplus\phi_2(x).
$$

### 乘积

$$
k(x,z)=k_1(x,z)k_2(x,z).
$$

课件 p7 的课堂证明可以补成：

$$
\phi(x)=\phi_1(x)\otimes\phi_2(x),
$$

因为

$$
\begin{aligned}
\langle\phi_1(x)\otimes\phi_2(x),
\phi_1(z)\otimes\phi_2(z)\rangle
&=
\langle\phi_1(x),\phi_1(z)\rangle
\langle\phi_2(x),\phi_2(z)\rangle\\
&=k_1(x,z)k_2(x,z).
\end{aligned}
$$

从核矩阵角度，乘积核的 Gram 矩阵是

$$
K_1\circ K_2,
$$

即逐元素乘积。Schur 乘积定理保证两个正半定矩阵的逐元素乘积仍然正半定。

### 指数核练习

若 $k$ 是核，则

$$
e^{k(x,z)}
=
\sum_{m=0}^{\infty}
\frac{k(x,z)^m}{m!}
$$

也是核，因为：

- 每个 $k^m$ 都是乘积核；
- 系数 $1/m!$ 非负；
- 非负加权和仍是核。

### 非齐次多项式核

$$
\boxed{
k(x,z)=
(c+\langle x,z\rangle)^d,
\qquad c\ge0,\ d\in\mathbb N.
}
$$

常数核与线性核之和仍是核，再使用乘积封闭性即可。

---
<!-- FIRST_VERSION_SECTION_16_END -->

## Section 17 — 核矩阵与一般输入空间（Kernel Matrices and General Input Spaces）

> **术语（Terminology）**：核矩阵（Kernel Matrix）；结构化输入（Structured Inputs）；Hadamard 乘积（Hadamard Product）

<!-- FIRST_VERSION_SECTION_17_START -->
## 17. 核矩阵与一般输入空间（p8–9）

核方法并不要求输入一定是欧氏向量。输入可以是：

- 字符串；
- 集合；
- 图；
- 群元素；
- 概率分布；
- 其他结构对象。

只要能在其上定义合法的正半定核，就可以使用核学习方法。

对样本 $x_1,\ldots,x_m$：

$$
K_{ij}=k(x_i,x_j).
$$

合法核的定义可以完全通过核矩阵表达：

> 对任意样本个数、任意有限输入集合，由 $k$ 生成的 Gram 矩阵都必须正半定。

用矩阵重新证明封闭性：

- 线性核：

  $$
  K=XX^\top\succeq0;
  $$

- 非负缩放：

  $$
  K\succeq0\Rightarrow cK\succeq0;
  $$

- 求和：

  $$
  K_1,K_2\succeq0
  \Rightarrow K_1+K_2\succeq0;
  $$

- 乘积：

  $$
  K_1\circ K_2\succeq0.
  $$

核方法用 $N\times N$ 的 Gram 矩阵取代显式特征矩阵。这避开了特征维数，却不等于没有代价：存储核矩阵通常需要 $O(N^2)$，直接求解线性系统可能需要 $O(N^3)$。

---
<!-- FIRST_VERSION_SECTION_17_END -->

## Section 18 — 平移不变核（Translation-Invariant Kernels）

> **术语（Terminology）**：平移不变核（Translation-Invariant Kernel）；Gaussian 径向基核（Gaussian RBF Kernel）；余弦核（Cosine Kernel）

<!-- FIRST_VERSION_SECTION_18_START -->
## 18. 平移不变核（p10）

平移不变核具有形式

$$
K(x,z)=\psi(x-z).
$$

也就是说，同时平移两个输入不会改变相似度：

$$
K(x+a,z+a)=K(x,z).
$$

### 余弦核

一维情况下：

$$
K(x,z)=\cos(x-z).
$$

利用

$$
\cos(x-z)
=
\cos x\cos z+\sin x\sin z,
$$

可以取

$$
\phi(x)=(\cos x,\sin x).
$$

所以余弦差值是一个核。

### Gaussian RBF

$$
\boxed{
K(x,z)=e^{-\gamma\|x-z\|^2},
\qquad \gamma>0.
}
$$

它只依赖两点间距离。距离越近，核值越接近 1；距离越远，核值越接近 0。

---
<!-- FIRST_VERSION_SECTION_18_END -->

## Section 19 — Bochner 定理与频域解释（Bochner’s Theorem and the Spectral View）

> **术语（Terminology）**：Bochner 定理（Bochner’s Theorem）；Fourier 变换（Fourier Transform）；谱测度（Spectral Measure）；Mercer 分解（Mercer Decomposition）

<!-- FIRST_VERSION_SECTION_19_START -->
## 19. Bochner 定理（p11）

Bochner 定理完整刻画连续平移不变核。

设

$$
K(x,z)=\psi(x-z),
\qquad x,z\in\mathbb R^n.
$$

那么 $K$ 是正定核，当且仅当存在有限非负 Borel 测度 $\mu$，使

$$
\boxed{
\psi(t)
=
\int_{\mathbb R^n}
e^{i\langle t,\omega\rangle}
\,d\mu(\omega).
}
$$

换句话说：

> 连续平移不变正定核，恰好是非负测度的 Fourier 变换。

容易证明的一边是：

$$
\begin{aligned}
\sum_{i,j}\overline c_i c_jK(x_i,x_j)
&=
\int
\sum_{i,j}\overline c_i c_j
e^{i\langle x_j-x_i,\omega\rangle}
d\mu(\omega)\\
&=
\int
\left|
\sum_jc_je^{i\langle x_j,\omega\rangle}
\right|^2
d\mu(\omega)\\
&\ge0.
\end{aligned}
$$

所以只要频谱测度非负，就得到正半定核。

Gaussian RBF 的 Fourier 变换仍是非负的 Gaussian，因此是合法核。

也可以代数证明：

$$
e^{-\gamma\|x-z\|^2}
=
e^{-\gamma\|x\|^2}
e^{-\gamma\|z\|^2}
e^{2\gamma x^\top z},
$$

再使用指数核和乘积核的封闭性。

---
<!-- FIRST_VERSION_SECTION_19_END -->

### 补充与深化（Supplement and Deepening）

> 来源：ltfp §7.2–7.3，PDF p.197–212；MLbookSol p.42–44。

#### 2. Mercer 谱分解

在适当紧致性和可积条件下：

$$
k(x,z)=\sum_{m=1}^{\infty}\mu_m e_m(x)e_m(z),
\qquad
\mu_m\ge0.
$$

若

$$
f=\sum_m a_m e_m,
$$

则

$$
\boxed{
\|f\|_{\mathcal H}^2
=
\sum_m\frac{a_m^2}{\mu_m}.
}
$$

因此：

- 大 $\mu_m$：该函数方向代价低；
- 小 $\mu_m$：该方向受到强烈正则化。

核不仅定义样本相似度，也定义了对不同函数方向的先验偏好。

#### 3. 平移不变核的频域范数

若

$$
k(x,z)=q(x-z),
$$

Bochner 定理要求 $q$ 的 Fourier 频谱非负。相应 RKHS 范数为

$$
\boxed{
\|f\|_{\mathcal H}^2
=
\frac1{(2\pi)^d}
\int_{\mathbb R^d}
\frac{|\widehat f(\omega)|^2}
{\widehat q(\omega)}
\,d\omega.
}
$$

所以 $\widehat q(\omega)$ 越小，对该频率的惩罚越大。

这给常见核一个清楚的平滑性解释：

- Laplacian/exponential 核：对应有限阶 Sobolev 光滑性；
- Matérn 核：参数直接控制 Sobolev 光滑阶数；
- Gaussian 核：频谱衰减极快，会强烈惩罚高频，RKHS 更小、函数更平滑。

“核更光滑”通常意味着允许的 RKHS 更小、先验更强，而不是函数空间更大。

---

## Section 20 — 其他核函数（Other Kernel Functions）

> **术语（Terminology）**：Laplacian 核（Laplacian Kernel）；概率核（Probability Kernel）；Jaccard 核（Jaccard Kernel）

<!-- FIRST_VERSION_SECTION_20_START -->
## 20. 其他核函数（p12）

课件先问：

$$
\frac{1}{1+\|x-z\|}
$$

是不是核？以及

$$
e^{-\gamma\|x-z\|}
$$

是不是核？

在欧氏空间中，第二个是 Laplacian 核，是正定的。第一个可表示为

$$
\frac1{1+r}
=
\int_0^\infty e^{-t}e^{-tr}\,dt.
$$

因此

$$
\frac{1}{1+\|x-z\|}
=
\int_0^\infty
e^{-t}
e^{-t\|x-z\|}
\,dt,
$$

它是 Laplacian 核的非负混合，所以也是核。

### 概率核

在概率空间上，对事件 $A,B$ 定义

$$
\boxed{
k(A,B)
=
P(A\cap B)-P(A)P(B).
}
$$

这是两个指示变量的协方差。令

$$
\phi(A)(\omega)
=
\mathbf1_A(\omega)-P(A),
$$

则

$$
\langle\phi(A),\phi(B)\rangle_{L^2(P)}
=
P(A\cap B)-P(A)P(B).
$$

### 条件期望核

$$
k(x,z)
=
\mathbb E_c[p(x\mid c)p(z\mid c)].
$$

把

$$
\phi(x)(c)=p(x\mid c)
$$

看成关于潜变量 $c$ 的 $L^2$ 函数，就得到内积表示。

### B-spline 核

课件列出

$$
B_{2n+1}(x-z).
$$

它可以通过平移不变核的 Fourier 非负性验证。

### Jaccard 核

对非空集合 $A,B$：

$$
\boxed{
k(A,B)=\frac{|A\cap B|}{|A\cup B|}.
}
$$

它衡量集合的交并比，常用于集合相似度。

课件还提到：

- 字符串核；
- 图核；
- 群上的核；
- 不变核；
- 其他结构化对象上的核。

这体现了核方法的一个重要优势：只要能定义合适的正定相似性，就不必先把对象粗暴地编码成固定长度向量。

---
<!-- FIRST_VERSION_SECTION_20_END -->

### MLbookSol 中需注意的笔误

2. p.43 购物篮分类器若预测式为

$$
\operatorname{sign}(2\langle w,\phi(x)\rangle-1),
$$

增广参数应为

$$
(2w,-1),
$$

不是 $(2w,1)$。

3. p.44 最近质心核展开中，负类核平均前应是减号：

$$
\frac1{m_+}\sum_{y_i=1}K(x,x_i)
-
\frac1{m_-}\sum_{y_i=-1}K(x,x_i).
$$

## Section 21 — 验证核函数可能很困难（Why Kernel Validation Can Be Difficult）

> **术语（Terminology）**：BMV 猜想（BMV Conjecture）；Laplace 变换（Laplace Transform）；微分熵（Differential Entropy）

<!-- FIRST_VERSION_SECTION_21_START -->
## 21. 验证一个函数是核可能非常困难（p13）

课件给出两个高级例子，目的不是要求现场证明，而是说明核的正定性可能涉及很深的数学。

### BMV 相关例子

若 $A$ 是 Hermitian 矩阵，$B\succeq0$，则

$$
K(x,y)
=
\operatorname{tr}
\exp(A-(x+y)B)
$$

是 $\mathbb R_+\times\mathbb R_+$ 上的核。

这与 1979–2011 年间的 BMV 猜想有关。其直觉是：若

$$
f(t)=\operatorname{tr}e^{A-tB}
$$

能写成非负测度的 Laplace 变换

$$
f(t)=\int e^{-\lambda t}\,d\mu(\lambda),
$$

则

$$
f(x+y)
=
\int
e^{-\lambda x}e^{-\lambda y}
\,d\mu(\lambda),
$$

直接呈现为内积。

### 微分熵例子

定义

$$
h(t)=\operatorname{Ent}(X+\sqrt t\,Z),
\qquad Z\sim N(0,1),
$$

其中微分熵为

$$
\operatorname{Ent}(U)
=
-\int f(u)\log f(u)\,du.
$$

课件询问

$$
K(t,s)=h'(t+s)
$$

是否是 $\mathbb R_+\times\mathbb R_+$ 上的核，并把该问题标注为仍开放。

本页的核心结论是：核函数的正定性并不总能靠简单展开验证，有时会连接矩阵分析、Fourier/Laplace 变换和信息论中的深层问题。
<!-- FIRST_VERSION_SECTION_21_END -->

## Section 22 — 核化回归的推导（Derivation of Kernelized Regression）

> **术语（Terminology）**：核化回归（Kernelized Regression）；原始—对偶表示（Primal–Dual Representation）；Nyström 近似（Nyström Approximation）

<!-- FIRST_VERSION_SECTION_22_START -->
---

# L6 第二部分：核岭回归

## 22. 核化回归的推导（p14–16）

p14 是“Back to using Kernels”的过渡页。

考虑非线性特征上的岭回归：

$$
\boxed{
\min_w
\frac1N\sum_{i=1}^N
\bigl(y_i-w^\top\phi(x_i)\bigr)^2
+
\lambda\|w\|^2.
}
$$

对 $w$ 求导：

$$
-\frac2N
\sum_i
\bigl(y_i-w^\top\phi(x_i)\bigr)\phi(x_i)
+
2\lambda w=0.
$$

因此

$$
\lambda w
=
\frac1N
\sum_i
\bigl(y_i-w^\top\phi(x_i)\bigr)\phi(x_i).
$$

定义

$$
N\lambda\alpha_i
=
y_i-w^\top\phi(x_i).
$$

于是

$$
\boxed{
w=\sum_i\alpha_i\phi(x_i).
}
$$

这和 SVM 一样：最优解位于训练特征向量的张成空间中。

对新输入 $x$：

$$
\begin{aligned}
\hat f(x)
&=
w^\top\phi(x)\\
&=
\sum_i\alpha_i
\langle\phi(x_i),\phi(x)\rangle\\
&=
\sum_i\alpha_i k(x_i,x).
\end{aligned}
$$

### 怎样求 $\alpha$？

由

$$
N\lambda\alpha_i
=
y_i-w^\top\phi(x_i)
$$

和

$$
w=\sum_j\alpha_j\phi(x_j),
$$

得到

$$
N\lambda\alpha_i
=
y_i-\sum_j\alpha_jk(x_j,x_i).
$$

定义

$$
K_{ij}=k(x_i,x_j),
$$

则

$$
N\lambda\alpha=y-K\alpha.
$$

所以

$$
(K+N\lambda I)\alpha=y,
$$

即

$$
\boxed{
\alpha=(K+N\lambda I)^{-1}y.
}
$$

最终预测器为

$$
\boxed{
\hat f(x)
=
\sum_{i=1}^N
\alpha_i k(x_i,x)
=
k_x^\top(K+N\lambda I)^{-1}y,
}
$$

其中

$$
k_x=(k(x_1,x),\ldots,k(x_N,x))^\top.
$$

### KRR 与核 SVM 的系数形式

- SVM：

  $$
  w=\sum_i\alpha_i y_i\phi(x_i);
  $$

- KRR：

  $$
  w=\sum_i\alpha_i\phi(x_i).
  $$

KRR 的标签信息已经包含在

$$
\alpha=(K+N\lambda I)^{-1}y
$$

里面。

---
<!-- FIRST_VERSION_SECTION_22_END -->

### 补充与深化（Supplement and Deepening）

> 来源：ltfp §7.4、§7.6，PDF p.212–236；MLbookSol p.42。

#### 1. 表示形式

对

$$
\min_{f\in\mathcal H}
\frac1n\sum_i(y_i-f(x_i))^2
+\lambda\|f\|_{\mathcal H}^2,
$$

解可以写成

$$
f(x)=\sum_i\alpha_i k(x,x_i),
$$

一个规范系数解是

$$
\boxed{
\alpha=(K+n\lambda I)^{-1}y.
}
$$

若 $K$ 奇异，$\alpha$ 的表示可能不唯一：加入 $v\in\ker K$ 不改变函数。但正则化后的预测函数是唯一的。

#### 3. 核近似

直接形成和分解 $K\in\mathbb R^{n\times n}$ 通常需要 $O(n^2)$ 内存和 $O(n^3)$ 时间。

Nyström 近似选取 $m$ 列：

$$
K
\approx
K(:,I)K(I,I)^{-1}K(I,:),
$$

成本约为 $O(nm^2)$。

随机特征利用

$$
k(x,z)
=
\mathbb E_v[\phi(x,v)\phi(z,v)]
$$

构造

$$
\tilde k(x,z)
=
\frac1m\sum_{j=1}^m
\phi(x,v_j)\phi(z,v_j).
$$

区别是：

- Nyström：依赖已观察的训练输入；
- 随机特征：可以在看数据之前采样。

### MLbookSol 中需注意的笔误

1. p.42 的 KRR 目标若写成

$$
\lambda\alpha^\top G\alpha
+\frac1{2m}\|G\alpha-y\|^2,
$$

严格求导会得到 $G+2m\lambda I$，而不是 $G+m\lambda I$。差异来自正则项是否带 $1/2$；结构不变，但必须统一 $\lambda$ 的尺度。

## Section 23 — 核岭回归的偏差—方差（KRR Bias–Variance Analysis）

> **术语（Terminology）**：核岭回归（Kernel Ridge Regression, KRR）；平滑矩阵（Smoothing Matrix）；有效维数（Effective Dimension）；随机 Fourier 特征（Random Fourier Features, RFF）

<!-- FIRST_VERSION_SECTION_23_START -->
## 23. 核岭回归的偏差—方差分解（p17）

设训练输出向量为

$$
y=z+\varepsilon,
\qquad
z=\mathbb E[y],
\qquad
\varepsilon\sim\mathcal N(0,C).
$$

KRR 在训练点上的拟合值为

$$
\hat z
=
K(K+n\lambda I)^{-1}y.
$$

定义平滑矩阵

$$
S_\lambda=K(K+n\lambda I)^{-1}.
$$

于是

$$
\hat z=S_\lambda y.
$$

预测误差分解：

$$
\frac1n
\mathbb E_\varepsilon\|\hat z-z\|^2
=
\frac1n
\|\mathbb E\hat z-z\|^2
+
\frac1n
\operatorname{tr}
\operatorname{Var}(\hat z).
$$

由于

$$
I-K(K+n\lambda I)^{-1}
=
n\lambda(K+n\lambda I)^{-1},
$$

得到平方偏差项

$$
\boxed{
\operatorname{bias}(K)
=
n\lambda^2
z^\top
(K+n\lambda I)^{-2}
z.
}
$$

这里课件的 “bias” 实际上指归一化后的平方偏差。

方差项为

$$
\boxed{
\operatorname{variance}(K)
=
\frac1n
\operatorname{tr}
\left[
CK^2(K+n\lambda I)^{-2}
\right].
}
$$

若

$$
K=U\operatorname{diag}(\kappa_r)U^\top,
$$

则每个核特征方向的信号收缩系数为

$$
\frac{\kappa_r}{\kappa_r+n\lambda}.
$$

随着 $\lambda$ 增大：

- 信号保留比例下降；
- 偏差因子

  $$
  \left(
  \frac{n\lambda}{\kappa_r+n\lambda}
  \right)^2
  $$

  增大；

- 噪声传递因子

  $$
  \left(
  \frac{\kappa_r}{\kappa_r+n\lambda}
  \right)^2
  $$

  减小。

这就是核岭回归中的偏差—方差折中。

---
<!-- FIRST_VERSION_SECTION_23_END -->

### 补充与深化（Supplement and Deepening）

> 来源：ltfp §7.4、§7.6，PDF p.212–236；MLbookSol p.42。

#### 2. KRR 也是标签的线性平滑器

令

$$
q(x)=
(k(x,x_1),\ldots,k(x,x_n))^\top.
$$

则

$$
\hat f(x)
=
q(x)^\top(K+n\lambda I)^{-1}y
=
\sum_iw_i(x)y_i,
$$

其中

$$
w(x)=(K+n\lambda I)^{-1}q(x).
$$

训练点预测为

$$
\hat y=H_\lambda y,
\qquad
H_\lambda=K(K+n\lambda I)^{-1}.
$$

与 Nadaraya–Watson 不同：

- KRR 权重可以为负；
- 权重一般不和为 1；
- 负权重允许高阶抵消，因此能够利用更高的目标光滑度。

#### 4. Population effective dimension

在随机设计中，定义协方差算子

$$
\Sigma=\mathbb E[\phi(X)\otimes\phi(X)].
$$

它连接 RKHS 范数与预测误差：

$$
\|g\|_{L^2(p)}^2
=
\langle g,\Sigma g\rangle_{\mathcal H}.
$$

有效维数为

$$
\boxed{
\mathcal N(\lambda)
=
\operatorname{tr}
\bigl[
\Sigma(\Sigma+\lambda I)^{-1}
\bigr].
}
$$

KRR 风险的主要结构可以写成

$$
\mathbb E\|\hat f_\lambda-f_*\|_{L^2(p)}^2
\lesssim
\frac{\sigma^2}{n}\mathcal N(\lambda)
+
A(\lambda,f_*),
$$

其中

$$
A(\lambda,f_*)
=
\inf_{f\in\mathcal H}
\left\{
\|f-f_*\|_{L^2(p)}^2
+\lambda\|f\|_{\mathcal H}^2
\right\}.
$$

第一项是估计/方差，第二项是正则化逼近偏差。

#### 5. KRR 的光滑度自适应率

若核特征值满足 Sobolev 型衰减

$$
\mu_j\asymp j^{-2s/d},
$$

且目标函数具有 $t$ 阶光滑度，则典型地

$$
\mathcal N(\lambda)
\asymp
\lambda^{-d/(2s)},
\qquad
A(\lambda,f_*)\asymp\lambda^{t/s}.
$$

平衡二者得到

$$
\boxed{
\lambda_{\rm opt}
\asymp
n^{-2s/(2t+d)}
}
$$

和

$$
\boxed{
\mathbb E\|\hat f-f_*\|^2
\asymp
n^{-2t/(2t+d)}.
}
$$

当 $t=1$ 时恢复局部平均的

$$
n^{-2/(d+2)};
$$

目标越光滑，KRR 可以得到更快的收敛率。这正是“核方法适应目标光滑度”的含义。

---

## Section 24 — 近零正则化、最小范数与双下降（Ridgeless Interpolation and Double Descent）

> **术语（Terminology）**：无正则插值（Ridgeless Interpolation）；最小范数插值器（Minimum-Norm Interpolator）；隐式偏置（Implicit Bias）；双下降（Double Descent）

<!-- FIRST_VERSION_SECTION_24_START -->
## 24. 为什么有时 $\lambda\approx0$ 最好（p18）

课件展示 Liang 与 Rakhlin 2018 的 MNIST 数字对分类实验，使用 Gaussian kernel。测试的数字对包括：

$$
[2,5],\ [2,9],\ [3,6],\ [3,8],\ [4,7].
$$

图中测试误差在 $\lambda\approx0$ 时最低，随后随 $\lambda$ 增大而上升。

这表面上与“必须正则化才能防止过拟合”矛盾，但关键在于：

$$
\lambda=0
$$

并不一定意味着模型没有任何偏好。

当核矩阵可插值训练数据时，算法往往选择：

- 最小 RKHS 范数插值解；
- 或由伪逆、求解器隐式选择的最小范数解。

因此即使没有显式 ridge，仍可能存在强烈的隐式正则化。

在高维、核谱和数据结构合适时，插值噪声不一定造成很大的测试误差，这被称为 benign overfitting 或 ridgeless regression。

但图并不表示“正则化永远无用”，而是说明：

> 最佳 $\lambda$ 依赖数据、核、噪声和算法；经典 U 形图不是所有高维问题的普遍定律。
<!-- FIRST_VERSION_SECTION_24_END -->

### 补充与深化（Supplement and Deepening）

> 来源：ltfp §12.1–12.2，PDF p.359–375。

#### 1. 从零初始化的 GD 选择最小范数插值器

当 $d>n$、$XX^\top$ 可逆时，方程

$$
X\theta=y
$$

有无穷多个解。梯度下降从

$$
\theta_0=0
$$

开始，其迭代始终留在 $X$ 的行空间中，并收敛到

$$
\boxed{
\theta^\dagger
=
X^\top(XX^\top)^{-1}y.
}
$$

这是所有插值解中欧氏范数最小的解。核版本对应最小 RKHS 范数插值器。

因此“$\lambda=0$ 仍可能泛化”依赖于算法选中了哪个插值器，并不是所有零训练误差解都具有良好泛化。

#### 2. 双下降的精确例子

设

$$
x\sim\mathcal N(0,I_d),
\qquad
y=x^\top\theta_*+\epsilon,
\qquad
\operatorname{Var}(\epsilon)=\sigma^2.
$$

对最小范数 ERM：

当 $d\le n-2$ 时，

$$
\mathbb E[R(\hat\theta)-R^*]
=
\frac{\sigma^2d}{n-d-1}.
$$

当 $d\ge n+2$ 时，

$$
\mathbb E[R(\hat\theta)-R^*]
=
\frac{\sigma^2n}{d-n-1}
+
\|\theta_*\|^2\frac{d-n}{d}.
$$

在 $d\approx n$ 的插值阈值附近，逆矩阵方差爆炸；越过阈值后，方差再次下降，但出现投影偏差，从而形成第二次下降。

若针对每个 $d$ 重新调节 Ridge 的 $\lambda$，双下降峰值通常会显著减弱，甚至消失。

---

## Section 25 — 再生核希尔伯特空间（Reproducing Kernel Hilbert Space, RKHS）

> **术语（Terminology）**：再生核希尔伯特空间（Reproducing Kernel Hilbert Space, RKHS）；再生性质（Reproducing Property）；点值评价（Point Evaluation）

<!-- FIRST_VERSION_SECTION_25_START -->
---

# L6 第三部分：RKHS 与表示定理

## 25. 什么是 RKHS（p19）

RKHS 是 reproducing kernel Hilbert space，再生核 Hilbert 空间。

设 $\mathcal H$ 是函数

$$
f:\mathcal X\to\mathbb R
$$

组成的 Hilbert 空间。若存在核 $k$，使对任意 $f\in\mathcal H$：

$$
\boxed{
\langle f,k(\cdot,x)\rangle_{\mathcal H}
=
f(x),
}
$$

则称 $\mathcal H$ 为 RKHS。

这叫再生性质：函数在点 $x$ 的取值，可以通过与 $k(\cdot,x)$ 做内积得到。

令 $f=k(\cdot,x')$，可得

$$
\boxed{
\langle k(\cdot,x'),k(\cdot,x)\rangle_{\mathcal H}
=
k(x',x).
}
$$

因此一个自然的特征映射是

$$
\phi(x)=k(\cdot,x).
$$

而且

$$
\mathcal H
=
\overline{
\operatorname{span}
\{k(\cdot,x):x\in\mathcal X\}
}.
$$

也就是说，RKHS 中的函数可以由核截面 $k(\cdot,x)$ 的有限线性组合及其极限构造出来。

---
<!-- FIRST_VERSION_SECTION_25_END -->

### 补充与深化（Supplement and Deepening）

> 来源：ltfp §7.2–7.3，PDF p.197–201。

#### 3. 从核直接构造 RKHS

先定义

$$
\mathcal H_0
=
\left\{
\sum_i\alpha_i k(\cdot,x_i)
\right\}
$$

及内积

$$
\left\langle
\sum_i\alpha_i k(\cdot,x_i),
\sum_j\beta_j k(\cdot,z_j)
\right\rangle
=
\sum_{i,j}\alpha_i\beta_jk(x_i,z_j).
$$

取完备化后得到 RKHS，并自动满足再生性质

$$
\boxed{
\langle k(\cdot,x),f\rangle_{\mathcal H}
=f(x).
}
$$

由 Cauchy–Schwarz：

$$
\boxed{
|f(x)|
\le
\|f\|_{\mathcal H}\sqrt{k(x,x)}.
}
$$

所以 RKHS 范数控制意味着逐点函数值受到控制。

普通 $L^2$ 空间不是 RKHS，因为 $L^2$ 中函数只在“几乎处处相等”意义下定义，单点取值不是连续泛函。

---

## Section 26 — 表示定理（Representer Theorem）

> **术语（Terminology）**：表示定理（Representer Theorem）；正交分解（Orthogonal Decomposition）；核截面（Kernel Section）

<!-- FIRST_VERSION_SECTION_26_START -->
## 26. Representer Theorem：表示定理（p20）

考虑一般的正则化经验风险：

$$
\min_{f\in\mathcal H}
R_n\bigl(
f(x^{(1)}),\ldots,f(x^{(n)})
\bigr)
+
\lambda\Omega(f).
$$

若正则器随着 Hilbert 范数单调不减，即

$$
\|f\|_{\mathcal H}>\|g\|_{\mathcal H}
\quad\Longrightarrow\quad
\Omega(f)\ge\Omega(g),
$$

则至少存在一个最优解具有有限展开：

$$
\boxed{
f(\cdot)
=
\sum_{i=1}^n
\alpha_i k(\cdot,x^{(i)}).
}
$$

若 $\Omega$ 对范数严格单调，则所有最优解都具有这种形式。

### 为什么成立？

令

$$
S=
\operatorname{span}
\{
k(\cdot,x^{(1)}),\ldots,k(\cdot,x^{(n)})
\}.
$$

任意 $f\in\mathcal H$ 可以正交分解为

$$
f=f_\parallel+f_\perp,
\qquad
f_\parallel\in S,\quad f_\perp\perp S.
$$

对每个训练点：

$$
\begin{aligned}
f(x^{(i)})
&=
\langle f,k(\cdot,x^{(i)})\rangle\\
&=
\langle f_\parallel,k(\cdot,x^{(i)})\rangle,
\end{aligned}
$$

因为

$$
\langle f_\perp,k(\cdot,x^{(i)})\rangle=0.
$$

所以删掉 $f_\perp$ 不会改变任何训练点上的预测，也不会改变损失。

但

$$
\|f\|^2
=
\|f_\parallel\|^2+\|f_\perp\|^2
\ge
\|f_\parallel\|^2.
$$

若正则器随范数单调，删掉 $f_\perp$ 不会让目标变差。因此最优解可以完全位于有限维空间 $S$ 中。

这解释了为什么一个原本可能无限维的优化问题，最终只需学习 $n$ 个系数。

两种等价写法是：

- 特征空间向量：

  $$
  w=\sum_i\alpha_i\phi(x_i);
  $$

- RKHS 函数：

  $$
  f(\cdot)=\sum_i\alpha_i k(x_i,\cdot).
  $$

p21 给出的阅读建议是课程教材中线性预测、logistic regression、SVM 与 kernel 的相关章节，以及 Moodle 上的 SVM/kernel notes。
<!-- FIRST_VERSION_SECTION_26_END -->

### 补充与深化（Supplement and Deepening）

> 来源：ltfp §7.2–7.3，PDF p.197–201。

#### 1. 更一般的表示定理

考虑目标

$$
\Psi\bigl(
f(x_1),\ldots,f(x_n),\|f\|_{\mathcal H}^2
\bigr),
$$

若 $\Psi$ 对最后一个范数变量严格递增，则最优解一定可以写成

$$
\boxed{
f=\sum_{i=1}^n\alpha_i k(\cdot,x_i).
}
$$

证明是把

$$
f=f_D+f_\perp
$$

分解为训练特征张成空间中的部分和正交部分。$f_\perp$ 不改变任何训练点预测，却会增加范数，所以最优时必须 $f_\perp=0$。

重要的是：表示定理本身不要求损失凸；凸性主要用于保证优化容易、解的唯一性和对偶分析。

#### 2. 最小范数插值版本

若存在满足

$$
f(x_i)=y_i
$$

的 RKHS 函数，则最小范数插值器满足

$$
f(\cdot)
=
\sum_i\alpha_i k(\cdot,x_i),
\qquad
K\alpha=y.
$$

这正是 ridgeless KRR 和最小范数隐式偏置的基础。

## Section 27 — 核密度估计（Kernel Density Estimation, KDE）

> **术语（Terminology）**：核密度估计（Kernel Density Estimation, KDE）；平滑核（Smoothing Kernel）；带宽（Bandwidth）；交叉验证（Cross-Validation）

<!-- FIRST_VERSION_SECTION_27_START -->
---

# L6 第四部分：KDE、核回归与 Attention

p22 是 “Kernels and Attention” 的过渡页。

## 27. 核密度估计 KDE（p23）

给定一维 iid 样本

$$
x_1,\ldots,x_n\sim f,
$$

希望估计未知密度 $f(x)$。

KDE 为

$$
\boxed{
\hat f_h(x)
=
\frac1n\sum_{i=1}^nK_h(x-x_i)
=
\frac1{nh}
\sum_{i=1}^n
K\left(\frac{x-x_i}{h}\right).
}
$$

其中：

- $K$ 是平滑窗口；
- $h>0$ 是带宽。

标准 KDE 通常要求

$$
\int K(u)\,du=1,
$$

并常取 $K(u)\ge0$。课件还给出零均值条件

$$
\int uK(u)\,du=0,
$$

它有助于消除一阶偏差项。

### 带宽的偏差—方差折中

- $h$ 很小：每个样本形成很窄的峰，能追踪局部细节，但曲线剧烈波动，低偏差、高方差；
- $h$ 很大：曲线平滑稳定，但可能抹掉真实结构，高偏差、低方差。

课件图中：

- $h=0.3$：较平滑；
- $h=0.1$：保留更多局部结构；
- $h=0.05$：明显过于波动。

实践中可用交叉验证或 plug-in 方法选择带宽。在一维且密度足够光滑时，典型渐近规律是

$$
h_{\mathrm{opt}}\propto n^{-1/5}.
$$

### 注意：“kernel”有不同含义

KDE 中的 $K$ 主要是平滑窗口，不要求一定是前面 RKHS 意义下的正半定 Mercer 核。

---
<!-- FIRST_VERSION_SECTION_27_END -->

## Section 28 — Nadaraya–Watson 核回归（Nadaraya–Watson Regression, NW）

> **术语（Terminology）**：Nadaraya–Watson 回归（Nadaraya–Watson Regression, NW）；局部平均（Local Averaging）；一致性（Consistency）；维数灾难（Curse of Dimensionality）

<!-- FIRST_VERSION_SECTION_28_START -->
## 28. Nadaraya–Watson 核回归（p24）

回归的目标是估计条件均值

$$
m(x)=\mathbb E[Y\mid X=x].
$$

由条件密度：

$$
\mathbb E[Y\mid X=x]
=
\int y f(y\mid x)\,dy
=
\int y\frac{f(x,y)}{f(x)}\,dy.
$$

使用乘积核估计联合密度：

$$
\hat f(x,y)
=
\frac1n
\sum_{i=1}^n
K_h(x-x_i)K_h(y-y_i),
$$

边缘密度估计为

$$
\hat f(x)
=
\frac1n
\sum_{i=1}^n
K_h(x-x_i).
$$

代入条件均值，并利用归一化、零均值核满足

$$
\int yK_h(y-y_i)\,dy=y_i,
$$

得到

$$
\boxed{
\hat m(x)
=
\frac{
\sum_iK_h(x-x_i)y_i
}{
\sum_jK_h(x-x_j)
}.
}
$$

定义

$$
w_i(x)
=
\frac{K_h(x-x_i)}
{\sum_jK_h(x-x_j)},
$$

则

$$
\boxed{
\hat m(x)=\sum_iw_i(x)y_i.
}
$$

若 $K\ge0$，这些权重非负且和为 1。因此它是一个加权最近邻估计器：

- 距离查询点越近的样本，权重越大；
- 预测是附近标签的加权平均。

---
<!-- FIRST_VERSION_SECTION_28_END -->

### 补充与深化（Supplement and Deepening）

> 来源：ltfp §6.2–6.5，PDF p.171–194；Attention：ltfp PDF p.294–295。

#### 1. NW 中的 kernel 不要求 PSD

Nadaraya–Watson 权重为

$$
w_i(x)
=
\frac{k(x,x_i)}
{\sum_jk(x,x_j)},
\qquad
w_i(x)\ge0,
\qquad
\sum_iw_i(x)=1.
$$

这里的 kernel 只要求逐点非负，目的是构造局部权重；它不需要满足 Gram 矩阵正半定。

这与 SVM/KRR 中的 Mercer 核是两套不同概念。Gaussian 恰好同时满足两类条件，并不意味着两种“核”定义相同。

#### 2. NW 的条件偏差—方差

若

$$
\hat f(x)=\sum_iw_i(x)y_i,
\qquad
f_*(x)=\mathbb E[Y\mid X=x],
$$

则条件于训练输入：

$$
\begin{aligned}
\mathbb E[
(\hat f(x)-f_*(x))^2\mid X_{1:n}
]
={}&
\left[
\sum_iw_i(x)(f_*(x_i)-f_*(x))
\right]^2\\
&+
\sum_iw_i(x)^2
\operatorname{Var}(Y_i\mid x_i).
\end{aligned}
$$

若 $f_*$ 是 $B$-Lipschitz 且噪声方差不超过 $\sigma^2$，则

$$
\le
B^2\sum_iw_i(x)\|x_i-x\|^2
+
\sigma^2\sum_iw_i(x)^2.
$$

权重越集中，偏差通常越小，但

$$
\sum_iw_i^2
$$

越大，方差越高。

#### 3. 带宽和维数灾难

对 $d$ 维 Lipschitz 回归函数：

$$
\operatorname{Variance}
\asymp
\frac{\sigma^2}{nh^d},
\qquad
\operatorname{Bias}^2
\asymp
B^2h^2.
$$

因此

$$
\boxed{
h_{\rm opt}\asymp n^{-1/(d+2)}
}
$$

以及

$$
\boxed{
\operatorname{Risk}
\asymp n^{-2/(d+2)}.
}
$$

一致性要求同时满足

$$
h\to0,
\qquad
nh^d\to\infty.
$$

若目标函数二阶光滑、核具有适当对称性，则平方偏差可改善为 $O(h^4)$，此时

$$
h_{\rm opt}\asymp n^{-1/(d+4)},
\qquad
\operatorname{Risk}\asymp n^{-4/(d+4)}.
$$

特别注意：

- 一维二阶光滑 KDE/NW 中常见 $h\asymp n^{-1/5}$；
- 仅假设 Lipschitz 的一维 NW 则是 $h\asymp n^{-1/3}$。

两种结论来自不同的光滑性假设，不能混用。

## Section 29 — 从 Nadaraya–Watson 到注意力机制（Attention）

> **术语（Terminology）**：注意力机制（Attention）；查询（Query）；键（Key）；值（Value）；缩放点积注意力（Scaled Dot-Product Attention）

<!-- FIRST_VERSION_SECTION_29_START -->
## 29. 从 Nadaraya–Watson 到 Attention

Attention 可以写成

$$
\boxed{
\operatorname{Attn}(q;K,V)
=
\frac{
\sum_j\kappa_\theta(q,k_j)v_j
}{
\sum_\ell\kappa_\theta(q,k_\ell)
}.
}
$$

Scaled dot-product attention 使用

$$
\kappa_\theta(q,k_j)
=
\exp\left(
\frac{q^\top k_j}{\sqrt d}
\right).
$$

对应关系为：

| 核回归 | Attention |
|---|---|
| 查询输入 $x$ | query $q$ |
| 训练输入 $x_j$ | key $k_j$ |
| 标签 $y_j$ | value $v_j$ |
| $K_h(x-x_j)$ | $\kappa_\theta(q,k_j)$ |
| 归一化核权重 | softmax 权重 |
| 标签加权平均 | value 加权平均 |

共同形式都是

$$
\text{输出}
=
\frac{
\sum_j
(\text{相似度})_j
(\text{值})_j
}{
\sum_j(\text{相似度})_j
}.
$$

### 但两者并不完全相同

- 核回归的相似度通常由固定距离和手工带宽决定；
- Attention 的 query、key 表示和相似性由数据学习；
- 核回归的值通常是标量标签，Attention 的 value 通常是向量；
- Attention 的 query 和 key 可以采用不同投影，因此相似性不一定对称；
- Attention 可以加入 mask、多头结构、位置编码和因果限制；
- 核回归通常是在 iid 样本上估计条件均值，Attention 是神经网络内部的上下文聚合机制；
- Attention 与核回归在形式上相似，但不一定是严格的 Mercer 核方法。

---

# 最后把两讲串起来

这两讲最重要的逻辑可以压缩为以下九步：

1. OLS 的解依赖 $(X^\top X)^{-1}$，可能不可逆或不稳定。
2. 岭回归通过 $\lambda I$ 稳定求解并压缩高方差方向。
3. 正则化引入偏差，但可能通过更大幅度降低方差而改善测试误差。
4. 线性模型表达能力不足时，可以先构造非线性特征 $\phi(x)$。
5. 显式特征可能组合爆炸甚至无限维。
6. SVM、KRR 等方法的最优解位于训练样本特征的张成空间中。
7. 因此算法只需计算

   $$
   \langle\phi(x_i),\phi(x_j)\rangle.
   $$

8. 用合法的正半定核

   $$
   k(x_i,x_j)
   =
   \langle\phi(x_i),\phi(x_j)\rangle
   $$

   替换内积，就得到核方法。
9. 表示定理从 RKHS 的一般理论解释了这种有限核展开为什么广泛成立。

还要始终区分两个独立的控制旋钮：

- 核 $k$：决定特征空间、相似性的含义和模型能表达什么；
- 正则化参数 $\lambda$：在选定特征空间中控制收缩程度和偏差—方差。

以及 “kernel” 的三种用法：

- SVM、KRR、RKHS 中：必须产生正半定 Gram 矩阵的内积核；
- KDE、Nadaraya–Watson 中：用于局部平滑的窗口，不一定是 Mercer 核；
- Attention 中：学习得到的相似度权重，与核回归形式相似，但不等同于传统核方法。
<!-- FIRST_VERSION_SECTION_29_END -->

### 补充与深化（Supplement and Deepening）

> 来源：ltfp §6.2–6.5，PDF p.171–194；Attention：ltfp PDF p.294–295。

#### 4. NW、KRR 和 Attention 的区别

| 方法 | 相似度条件 | 权重 | 主要调节量 |
|---|---|---|---|
| NW | 核窗逐点非负 | 非负、和为 1 | 带宽 $h$ |
| KRR | Gram 矩阵 PSD | 可为负、通常不和为 1 | 正则化 $\lambda$ |
| Attention | softmax 分数 | 非负、每行和为 1 | 学习出的 $Q,K,V$ |

Attention 的标准形式是

$$
\operatorname{Attention}(Q,K,V)
=
\operatorname{softmax}
\left(
\frac{QK^\top}{\sqrt d}
\right)V.
$$

它形式上类似“对 value 做归一化加权平均”，所以可与 NW 类比。但这只是结构类比：

- $Q,K,V$ 由数据和学习参数共同产生；
- query 与 key 映射可以不同；
- $QK^\top$ 不必对称；
- 它不必是一个 PSD Gram 矩阵。

因此不能把“Attention 是 NW/KRR”写成这两本书证明的定理。

---

### 整合说明

以上补充已经把两本书与原 L5/L6 笔记的对应内容合并，并保留了各公式适用的假设和来源边界。